In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from pathlib import Path
import time
import matplotlib.pyplot as plt
import copy

PARQUET_DIR = Path.home() / "projects" / "recsys" / "data" / "parquet"
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
torch.manual_seed(42)
np.random.seed(42)
print(f"device: {device}")

# load and split (same as day 7)
ratings = pd.read_parquet(PARQUET_DIR / "ratings_clean.parquet")
cutoff_ts = ratings["timestamp"].quantile(0.9)
train_df = ratings[ratings["timestamp"] < cutoff_ts].reset_index(drop=True)
val_df   = ratings[ratings["timestamp"] >= cutoff_ts].reset_index(drop=True)

user_to_idx = {u: i for i, u in enumerate(train_df["userId"].unique())}
movie_to_idx = {m: i for i, m in enumerate(train_df["movieId"].unique())}
n_users = len(user_to_idx)
n_movies = len(movie_to_idx)

def apply_mappings(df, user_map, movie_map):
    df = df.copy()
    df["user_idx"] = df["userId"].map(user_map)
    df["movie_idx"] = df["movieId"].map(movie_map)
    df = df.dropna(subset=["user_idx", "movie_idx"]).reset_index(drop=True)
    df["user_idx"] = df["user_idx"].astype(np.int32)
    df["movie_idx"] = df["movie_idx"].astype(np.int32)
    return df

train_df_mapped = apply_mappings(train_df, user_to_idx, movie_to_idx)
val_df_mapped   = apply_mappings(val_df, user_to_idx, movie_to_idx)
GLOBAL_MEAN = float(train_df_mapped["rating"].mean())

print(f"users: {n_users:,} | movies: {n_movies:,}")
print(f"train: {len(train_df_mapped):,} | val: {len(val_df_mapped):,}")


class MFWithBias(nn.Module):
    def __init__(self, n_users, n_movies, dim=32, global_mean=3.5):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, dim)
        self.movie_emb = nn.Embedding(n_movies, dim)
        self.user_bias = nn.Embedding(n_users, 1)
        self.movie_bias = nn.Embedding(n_movies, 1)
        self.global_mean = global_mean
        nn.init.normal_(self.user_emb.weight, std=0.05)
        nn.init.normal_(self.movie_emb.weight, std=0.05)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.movie_bias.weight)
    def forward(self, u, m):
        ue = self.user_emb(u); me = self.movie_emb(m)
        ub = self.user_bias(u).squeeze(1); mb = self.movie_bias(m).squeeze(1)
        return self.global_mean + ub + mb + (ue * me).sum(dim=1)

device: mps
users: 150,330 | movies: 45,058
train: 22,454,535 | val: 454,141


In [2]:
class RatingsDataset(Dataset):
    def __init__(self, df):
        self.users = torch.from_numpy(df["user_idx"].values.astype(np.int64))
        self.movies = torch.from_numpy(df["movie_idx"].values.astype(np.int64))
        self.ratings = torch.from_numpy(df["rating"].values.astype(np.float32))
    def __len__(self): return len(self.users)
    def __getitem__(self, i): return self.users[i], self.movies[i], self.ratings[i]

train_ds = RatingsDataset(train_df_mapped)
val_ds   = RatingsDataset(val_df_mapped)
BATCH_SIZE = 8192
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)

model = MFWithBias(n_users, n_movies, dim=32, global_mean=GLOBAL_MEAN).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-5)
loss_fn = nn.MSELoss()

best_val = float("inf")
best_state = None

for epoch in range(1, 6):
    model.train()
    t0 = time.time()
    for u, m, r in train_loader:
        u, m, r = u.to(device), m.to(device), r.to(device)
        optimizer.zero_grad()
        loss = loss_fn(model(u, m), r)
        loss.backward()
        optimizer.step()
    
    model.eval()
    se, n = 0.0, 0
    with torch.no_grad():
        for u, m, r in val_loader:
            u, m, r = u.to(device), m.to(device), r.to(device)
            pred = model(u, m)
            se += ((pred - r) ** 2).sum().item()
            n += len(r)
    val_rmse = (se / n) ** 0.5
    
    marker = ""
    if val_rmse < best_val:
        best_val = val_rmse
        best_state = copy.deepcopy(model.state_dict())
        marker = " *"
    print(f"epoch {epoch}/5  |  val rmse: {val_rmse:.4f}  |  {time.time()-t0:.1f}s{marker}")

model.load_state_dict(best_state)
print(f"\nbest val rmse: {best_val:.4f}")

epoch 1/5  |  val rmse: 0.8661  |  137.9s *
epoch 2/5  |  val rmse: 0.8632  |  136.0s *
epoch 3/5  |  val rmse: 0.8616  |  136.6s *
epoch 4/5  |  val rmse: 0.8628  |  137.2s
epoch 5/5  |  val rmse: 0.8617  |  139.3s

best val rmse: 0.8616


In [3]:
import math

def dcg(relevances):
    """discounted cumulative gain: sum of rel_i / log2(i+1) for i in [1..k]."""
    return sum(rel / math.log2(i + 2) for i, rel in enumerate(relevances))


def evaluate_ranking(model, train_df_mapped, val_df_mapped, n_users, n_movies,
                     k_list=(5, 10, 20), n_sample_users=1000, like_threshold=4.0,
                     seed=42):
    """
    for each sampled val user:
      1. get items they "liked" in val (rating >= like_threshold)
      2. score all unseen items (not in their train history)
      3. take top-k by predicted score
      4. compute recall@k and ndcg@k against their liked-in-val set
    
    returns dict with mean recall@k, ndcg@k, popularity-baseline recall@k.
    """
    rng = np.random.RandomState(seed)
    
    # build per-user sets
    train_by_user = train_df_mapped.groupby("user_idx")["movie_idx"].apply(set)
    val_by_user_liked = (
        val_df_mapped[val_df_mapped["rating"] >= like_threshold]
        .groupby("user_idx")["movie_idx"].apply(set)
    )
    
    # only evaluate on val users with at least 1 liked item
    eligible = list(val_by_user_liked.index)
    print(f"eligible val users (≥1 liked item): {len(eligible):,}")
    
    sample_users = rng.choice(eligible, size=min(n_sample_users, len(eligible)), replace=False)
    
    # popularity baseline: rank by training popularity
    pop_score = np.zeros(n_movies, dtype=np.float32)
    counts = train_df_mapped["movie_idx"].value_counts()
    pop_score[counts.index.values] = counts.values
    
    model.eval()
    all_movies_t = torch.arange(n_movies, dtype=torch.long, device=device)
    
    metrics = {f"recall@{k}": [] for k in k_list}
    metrics.update({f"ndcg@{k}": [] for k in k_list})
    metrics.update({f"pop_recall@{k}": [] for k in k_list})
    
    with torch.no_grad():
        for user_idx in sample_users:
            seen = train_by_user.get(user_idx, set())
            liked = val_by_user_liked[user_idx]
            
            # mask: True for candidate items (not seen in train)
            candidate_mask = np.ones(n_movies, dtype=bool)
            candidate_mask[list(seen)] = False
            
            # model scores for all movies
            user_t = torch.full((n_movies,), int(user_idx), dtype=torch.long, device=device)
            scores = model(user_t, all_movies_t).cpu().numpy()
            scores[~candidate_mask] = -np.inf  # exclude seen items
            
            # popularity scores
            pop_scores_masked = pop_score.copy()
            pop_scores_masked[~candidate_mask] = -np.inf
            
            for k in k_list:
                # model top-k
                top_k_model = np.argpartition(-scores, k)[:k]
                hits_model = liked.intersection(top_k_model.tolist())
                metrics[f"recall@{k}"].append(len(hits_model) / len(liked))
                
                # ndcg: relevance = 1 if liked, else 0; sort top-k by score
                top_k_sorted = top_k_model[np.argsort(-scores[top_k_model])]
                rels = [1 if m in liked else 0 for m in top_k_sorted]
                ideal_rels = [1] * min(k, len(liked))
                ndcg = dcg(rels) / dcg(ideal_rels) if ideal_rels else 0
                metrics[f"ndcg@{k}"].append(ndcg)
                
                # popularity baseline recall
                top_k_pop = np.argpartition(-pop_scores_masked, k)[:k]
                hits_pop = liked.intersection(top_k_pop.tolist())
                metrics[f"pop_recall@{k}"].append(len(hits_pop) / len(liked))
    
    return {key: float(np.mean(vals)) for key, vals in metrics.items()}


print("evaluating ranking metrics on 1000 sampled val users...")
t0 = time.time()
results = evaluate_ranking(model, train_df_mapped, val_df_mapped, n_users, n_movies,
                          k_list=(5, 10, 20), n_sample_users=1000)
print(f"done in {time.time()-t0:.1f}s\n")

print(f"{'metric':<20s} {'model':>10s} {'popularity':>12s} {'delta':>10s}")
print("-" * 56)
for k in (5, 10, 20):
    m_recall = results[f"recall@{k}"]
    p_recall = results[f"pop_recall@{k}"]
    delta = m_recall - p_recall
    print(f"{'recall@'+str(k):<20s} {m_recall:>10.4f} {p_recall:>12.4f} {delta:>+10.4f}")
print()
for k in (5, 10, 20):
    print(f"{'ndcg@'+str(k):<20s} {results[f'ndcg@'+str(k)]:>10.4f}")

evaluating ranking metrics on 1000 sampled val users...
eligible val users (≥1 liked item): 6,282
done in 8.9s

metric                    model   popularity      delta
--------------------------------------------------------
recall@5                 0.0201       0.0202    -0.0000
recall@10                0.0345       0.0301    +0.0044
recall@20                0.0520       0.0468    +0.0052

ndcg@5                   0.0783
ndcg@10                  0.0743
ndcg@20                  0.0728


the mf model with mse loss matches popularity baseline on recall@5 (0.0201 vs 0.0202) and edges popularity by 0.4-0.5pp on recall@10 and recall@20. mse on observed ratings does not optimize ranking from unseen candidates. fix: ranking losses with negative sampling (week 3, two-tower).